# 05 · Temporal post-processing and inference strategies

Bonus *Data temporality* and part of *Your pipeline* (inference strategy vs model change).
Uses the fold models of `yolo_1024` (best validated configuration, `03_yolo.ipynb`): no retraining.

- **Inference strategy (TTA):** Ultralytics test-time augmentation (flip + 2 scales) vs plain inference.
- **Temporal post-processing** (`temporal.py`): global camera-motion compensation (phase correlation),
  linking detections of consecutive frames into tracks by IoU, blending each score with the mean
  score of its track, and filling short gaps inside tracks.

Evaluation stays on the protocol frames (every 10th validation frame, `common.evaluate`), but the
temporal method sees **all** consecutive frames of the validation clips as context.
Parameters are selected leave-one-fold-out: fold *k* is scored with the setting that is best on the
other two folds, so the reported OOF number is not tuned on the frames it is measured on.

Two modes, so the GPU is used only where needed:
- `predict` (GPU, attach the output of the `03_yolo` version with `yolo_1024`): fold models predict every
  frame of their validation clips, plus TTA on the protocol frames; predictions are saved as outputs.
- `tune` (CPU is enough, attach the output of the `predict` run **and** the `03_yolo` version with the
  final `submission.csv`): temporal grid, leave-one-fold-out selection, logging, temporal test submission.

## 1. Run configuration

In [ ]:
MODE = "tune"              # "predict" (GPU) | "tune" (CPU)
SOURCE = "yolo_1024"       # experiment whose fold models are used
QUICK = False              # True: 1 clip per fold, few frames (code check only)

## 2. Setup

In [ ]:
import os, sys, shutil, subprocess, time, urllib.request
from pathlib import Path

os.environ["YOLO_VERBOSE"] = "False"
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--timeout", "60", "--retries", "5",
                    "ultralytics==8.4.203", "pycocotools"], check=True, timeout=1800)
    import ultralytics
os.environ["WANDB_MODE"] = "disabled"

REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks/"

def fetch(url, dst, timeout=60, tries=5):
    # download with a timeout and retries: a stalled GitHub request must not hang a commit for hours
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                Path(dst).write_bytes(r.read())
            return dst
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"{url}: {type(e).__name__}, retry in 30 s"); time.sleep(30)

SRC_FILES = ["common.py", "ul_dataset.py", "temporal.py", "splits.json"]
SRC = next((Path(p) for p in [".", "notebooks"] if all((Path(p) / f).exists() for f in SRC_FILES)), None)
if SRC is None:
    # GitHub is the source of truth; an attached copy is only a fallback when GitHub is unreachable
    SRC = Path("/tmp/hw2_src"); SRC.mkdir(exist_ok=True)
    for f in SRC_FILES:
        try:
            fetch(REPO_RAW + f, SRC / f, tries=3); print(f"{f}: GitHub")
        except Exception:
            found = [p for pat in ["*", "*/*", "*/*/*"] for p in sorted(Path("/kaggle/input").glob(f"{pat}/{f}"))
                     if "competitions" not in p.parts] if Path("/kaggle/input").exists() else []
            if not found:
                raise
            shutil.copy(found[0], SRC / f); print(f"{f}: {found[0]} (GitHub unreachable)")
sys.path.insert(0, str(SRC))

In [ ]:
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import common, ul_dataset as uld, temporal as T
importlib.reload(common); importlib.reload(uld); importlib.reload(T)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
RUN_ROOT = WORK / "ul_runs"
PRED_DIR = WORK / "temporal_preds"
IMG_DIR = WORK / "images"
RESULTS = WORK / ("results.csv" if not QUICK else "results_quick.csv")
for p in (RUN_ROOT, PRED_DIR, IMG_DIR):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "0" if torch.cuda.is_available() else "cpu"
if MODE == "predict" and not QUICK:
    assert torch.cuda.is_available(), "attach a GPU (Settings -> Accelerator)"

images, anns = common.load_gt(cache="/tmp/gt_cache.pkl")
splits = common.load_splits(SRC / "splits.json")
if QUICK:
    for f in splits["folds"]:
        f["val_clips"] = f["val_clips"][:1]
cfg = uld.Config(experiment=SOURCE, imgsz=1024, batch=8, pred_batch=16, device=DEVICE)
print("device:", DEVICE, "| mode:", MODE)

## 3. Predictions on consecutive frames (`MODE = "predict"`)
For each fold, the `yolo_1024` model of that fold (trained on the other folds' clips) predicts every
frame of the fold's validation clips; TTA predictions are made on the protocol frames only.

In [ ]:
if MODE == "predict":
    uld.restore_from_inputs(RUN_ROOT)
    for k in range(common.N_FOLDS):
        w = RUN_ROOT / SOURCE / f"fold{k}" / "final.pt"
        assert w.exists(), f"{w} missing: attach the 03_yolo output that trained {SOURCE}"
        model = uld.load_trained(cfg, w)
        allf = common.fold_images(images, splits, k, "val", 1)
        if QUICK:
            allf = allf.groupby("clip").head(25)
        t0 = time.time()
        p = uld.predict(model, allf["path"].tolist(), allf["img_id"].tolist(), cfg)
        p.rename(columns={"key": "img_id"}).to_parquet(PRED_DIR / f"fold{k}_all.parquet")
        ev = common.fold_images(images, splits, k, "val", common.EVAL_STRIDE)
        ev = ev[ev.img_id.isin(allf.img_id)]
        pt = uld.predict(model, ev["path"].tolist(), ev["img_id"].tolist(), cfg, augment=True)
        pt.rename(columns={"key": "img_id"}).to_parquet(PRED_DIR / f"fold{k}_tta.parquet")
        print(f"fold {k}: {len(allf)} frames, {len(p)} boxes, TTA on {len(ev)} frames, {(time.time() - t0) / 60:.1f} min")

## 4. Load predictions (`MODE = "tune"`)
Per-fold predictions from the attached `predict` output. Frame index: every frame of each fold's
validation clips (keys = `img_id`), used for linking and for mapping gap-filled boxes to frames.

In [ ]:
if MODE == "tune":
    src = {}
    for k in range(common.N_FOLDS):
        for kind in ["all", "tta"]:
            hit = [PRED_DIR / f"fold{k}_{kind}.parquet"] if (PRED_DIR / f"fold{k}_{kind}.parquet").exists() \
                else uld._input_files("/kaggle/input", f"temporal_preds/fold{k}_{kind}.parquet")
            assert hit, f"fold{k}_{kind}.parquet missing: attach the output of the predict run"
            src[(k, kind)] = pd.read_parquet(hit[0])
    frame_idx = {}
    for k in range(common.N_FOLDS):
        f = common.fold_images(images, splits, k, "val", 1)
        if QUICK:
            f = f[f.img_id.isin(src[(k, "all")].img_id.unique())]
        frame_idx[k] = f.rename(columns={"img_id": "key", "frame_id": "frame"})[["key", "clip", "frame", "path"]]
    print({k: len(v) for k, v in frame_idx.items()}, "frames per fold")

### 4.1 Camera motion
Global translation between consecutive frames (phase correlation on 4× downscaled grayscale frames).
Large shifts mean that without compensation a static object's boxes in consecutive frames may not overlap.

In [ ]:
if MODE == "tune":
    t0 = time.time()
    shifts = {}
    for k, f in frame_idx.items():
        for clip, g in f.sort_values("frame").groupby("clip"):
            shifts[clip] = T.frame_shifts(g["path"].tolist())
    mag = np.concatenate([np.hypot(*s[1:].T) for s in shifts.values()])
    print(f"{len(shifts)} clips, {(time.time() - t0) / 60:.1f} min | shift per frame (px): "
          f"median {np.median(mag):.1f}, p90 {np.percentile(mag, 90):.1f}, max {mag.max():.1f}")
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(np.clip(mag, 0, 40), bins=40); ax.set_xlabel("camera shift between consecutive frames, px (clipped at 40)")
    ax.set_ylabel("frame pairs"); fig.tight_layout(); fig.savefig(IMG_DIR / "05_camera_shift.png"); plt.show()

**Observations (camera motion):** TBD

## 5. Experiments
Every setting is evaluated per fold on the protocol frames. `none` is the per-frame baseline (must
reproduce the `yolo_1024` OOF result); `tta` is test-time augmentation without temporal processing.

In [ ]:
def per_fold(pred_by_fold):
    return common.evaluate_oof(pred_by_fold, images, anns, splits, common.EVAL_STRIDE)

def run_temporal(params):
    out = {}
    for k in range(common.N_FOLDS):
        p = src[(k, "all")].rename(columns={"img_id": "key"})
        sh = shifts if params.get("motion", True) else None
        kw = {a: b for a, b in params.items() if a != "motion"}
        r = T.apply(p, frame_idx[k], shifts=sh, **kw)
        out[k] = r.rename(columns={"key": "img_id"}).astype({"img_id": int})[["img_id", "cls", "score", "x", "y", "w", "h"]]
    return out

GRID = {"none": None, "tta": "tta",
        "rescore_nomotion": dict(alpha=0.5, window=5, motion=False)}
for a in [0.3, 0.5, 0.7]:
    for w in [2, 5, 10]:
        GRID[f"rescore_a{a}_w{w}"] = dict(alpha=a, window=w)
if QUICK:
    GRID = {k: GRID[k] for k in ["none", "tta", "rescore_nomotion", "rescore_a0.5_w5"]}

if MODE == "tune":
    table, preds_of = {}, {}
    t0 = time.time()
    for name, prm in GRID.items():
        if prm is None:
            pf = {k: src[(k, "all")] for k in range(common.N_FOLDS)}
        elif prm == "tta":
            pf = {k: src[(k, "tta")] for k in range(common.N_FOLDS)}
        else:
            pf = run_temporal(prm)
        m = per_fold(pf)
        table[name] = m
        print(f"{name:22s} OOF mAP {m.at['oof', 'mAP']:.4f}  AP_small {m.at['oof', 'AP_small']:.4f}  "
              f"[{(time.time() - t0) / 60:.1f} min]")

Gap filling and causal (past-only) scoring, on top of the best rescoring setting (by mean fold mAP).

In [ ]:
if MODE == "tune":
    resc = [n for n in table if n.startswith("rescore_a")]
    best = max(resc, key=lambda n: table[n].at["mean", "mAP"])
    base_prm = GRID[best]
    extra = {f"{best}_fill": dict(base_prm, fill=True, decay=0.5),
             f"{best}_causal": dict(base_prm, causal=True)}
    for name, prm in extra.items():
        GRID[name] = prm
        table[name] = per_fold(run_temporal(prm))
        print(f"{name:28s} OOF mAP {table[name].at['oof', 'mAP']:.4f}  AP_small {table[name].at['oof', 'AP_small']:.4f}")

### 5.1 Results and leave-one-fold-out selection
For every fold, the temporal setting with the best mean mAP on the **other two** folds is applied to
it; the pooled result of these choices is the honest OOF estimate of the temporal method.

In [ ]:
MAIN = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR500"]
if MODE == "tune":
    summ = pd.DataFrame({n: {**{f"{c}": m.at["oof", c] for c in MAIN},
                             **{f"fold{k}": m.at[k, "mAP"] for k in range(common.N_FOLDS)}}
                         for n, m in table.items()}).T
    summ["d_mAP_vs_none"] = summ["mAP"] - summ.at["none", "mAP"]
    display(summ.round(4))

    temporal_names = [n for n in table if n not in ("none", "tta")]
    choice = {}
    for k in range(common.N_FOLDS):
        others = [j for j in range(common.N_FOLDS) if j != k]
        choice[k] = max(temporal_names, key=lambda n: np.mean([table[n].at[j, "mAP"] for j in others]))
    print("leave-one-fold-out choice:", choice)
    lofo = {}
    for k, n in choice.items():
        lofo[k] = run_temporal(GRID[n])[k]
    m_lofo = per_fold(lofo)
    display(m_lofo[MAIN].round(4))

Per-size and per-class effect of the selected temporal setting vs per-frame predictions.

In [ ]:
if MODE == "tune":
    none_pc = common.evaluate_oof({k: src[(k, "all")] for k in range(common.N_FOLDS)}, images, anns, splits,
                                  common.EVAL_STRIDE, per_class=True)
    lofo_pc = common.evaluate_oof(lofo, images, anns, splits, common.EVAL_STRIDE, per_class=True)
    cols = ["AP_small", "AP_medium", "AP_large"] + [f"AP_{c}" for c in common.CLASSES]
    d = pd.DataFrame({"per-frame": none_pc.loc["oof", cols], "temporal (LOFO)": lofo_pc.loc["oof", cols]})
    d["delta"] = d.iloc[:, 1] - d.iloc[:, 0]
    display(d.round(4))
    fig, ax = plt.subplots(figsize=(9, 3))
    d["delta"].plot.bar(ax=ax, color=["tab:green" if v >= 0 else "tab:red" for v in d["delta"]])
    ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("Δ AP (temporal − per-frame)")
    fig.tight_layout(); fig.savefig(IMG_DIR / "05_temporal_delta.png"); plt.show()

**Observations (temporal):** TBD

Example track: raw vs rescored score of one long track in a validation clip (fold 0).

In [ ]:
if MODE == "tune":
    prm = GRID[choice[0]]
    p0 = src[(0, "all")].rename(columns={"img_id": "key"})
    r0 = T.apply(p0, frame_idx[0], shifts=shifts if prm.get("motion", True) else None,
                 **{a: b for a, b in prm.items() if a != "motion"})
    r0 = r0[r0["track"] >= 0]
if MODE == "tune" and len(r0) == 0:
    print("no tracks (no detection above min_score)")
elif MODE == "tune":
    lens = r0.groupby(["clip", "track"]).size()
    clip, tr = lens.idxmax()
    g = r0[(r0["clip"] == clip) & (r0["track"] == tr)].sort_values("frame")
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(g["frame"], g["score_raw"], ".", ms=3, label="per-frame score")
    ax.plot(g["frame"], g["score"], "-", lw=1.2, label="temporal score")
    ax.set_xlabel(f"frame ({clip}, class {common.CLASSES[int(g['cls'].iloc[0])]})"); ax.set_ylabel("score"); ax.legend()
    fig.tight_layout(); fig.savefig(IMG_DIR / "05_track_example.png"); plt.show()

## 6. Logging
TTA and the leave-one-fold-out temporal result are logged to `results.csv` next to `yolo_1024`.

In [ ]:
if MODE == "tune":
    info = dict(owner="B", model="yolo", train_stride=common.TRAIN_STRIDE, eval_stride=common.EVAL_STRIDE,
                imgsz=1024, epochs=12)
    logged = WORK / ("logged_quick" if QUICK else "logged")
    if not logged.exists():
        common.log_results(table["tta"], RESULTS, experiment=f"{SOURCE}_tta", notes="Ultralytics TTA (flip + scales)", **info)
        common.log_results(m_lofo, RESULTS, experiment=f"{SOURCE}_temporal",
                           notes="track rescoring, params chosen leave-one-fold-out: " + str(choice), **info)
        logged.touch()
    print(pd.read_csv(RESULTS).query("fold == 'oof'")[["experiment", "mAP", "AP50", "AP_small"]])

## 7. Temporal test submission
Applies the temporal setting chosen on all three folds to the final model's test predictions
(`submission.csv` of the attached `03_yolo` submit run). Test frames of a clip are consecutive.

In [ ]:
def parse_submission(path):
    sub = pd.read_csv(path, keep_default_na=False)
    rows = []
    for iid, s in zip(sub["image_id"], sub["PredictionString"]):
        v = s.split()
        for i in range(0, len(v), 6):
            rows.append((iid, int(v[i]), float(v[i + 1]), *map(float, v[i + 2:i + 6])))
    return pd.DataFrame(rows, columns=["image_id", "cls", "score", "x", "y", "w", "h"])

if MODE == "tune":
    final = max(temporal_names, key=lambda n: table[n].at["mean", "mAP"])
    print("setting for test:", final, GRID[final])
    hits = [p for p in uld._input_files("/kaggle/input", "submission.csv") if "temporal" not in p.name]
    if not hits:
        print("no submission.csv attached: attach the 03_yolo submit run output")
    else:
        test = common.load_test_index()
        tp = parse_submission(hits[0])
        tidx = test.rename(columns={"image_id": "key"})[["key", "clip", "frame", "path"]]
        prm = GRID[final]
        tsh = None
        if prm.get("motion", True):
            tsh = {c: T.frame_shifts(g.sort_values("frame")["path"].tolist()) for c, g in tidx.groupby("clip")}
        r = T.apply(tp.rename(columns={"image_id": "key"}), tidx, shifts=tsh,
                    **{a: b for a, b in prm.items() if a != "motion"})
        out = common.write_submission(r.rename(columns={"key": "image_id"}), test, WORK / "submission_temporal.csv")
        sample = WORK / "sample_from_index.csv"
        test[["image_id"]].assign(PredictionString="").to_csv(sample, index=False)
        print(f"source: {hits[0]}\n", common.check_submission(out, sample))

## 8. Conclusions
TBD